# Polarization with Jones matrices

*optics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/polarization_jones)

The polarization of light is a complex 2-vector $(E_x, E_y)$ and every polarizer or wave plate is a complex $2\times 2$ matrix. Light passes a wave plate and then a rotating polarizer: the $\mathbf E$-field ellipse and the transmitted intensity follow from einsum, reproducing Malus's law and turning linear into circular light.

**Vector form**

$$\mathbf E_{\text{out}} = P(\theta)\,W(\Gamma,\alpha)\,\mathbf E_{\text{in}},\quad P(\theta) = R(\theta)\begin{pmatrix}1&0\\0&0\end{pmatrix}R(-\theta),\ W = R(\alpha)\begin{pmatrix}1&0\\0&e^{i\Gamma}\end{pmatrix}R(-\alpha),\quad I = \mathbf E^\dagger\mathbf E$$

**Index (tensor) form**

$$E^{\text{out}}_i = P_{ij}\,W_{jk}\,E^{\text{in}}_k,\qquad I = E^{*}_i\,E_i,\qquad \mathcal E_i(t) = \operatorname{Re}\big(E_i\,e^{-i\omega t}\big)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $E_i$ | `e_in` | (2,) complex | Jones vector: complex amplitudes of $E_x$ and $E_y$ (the phase difference sets the shape) |
| $R_{ij}(\theta)$ | `rotation(theta)` | (2, 2) | rotates the lab frame onto the element's own axes |
| $P_{ij}(\theta)$ | `polarizer(theta)` | (2, 2) complex | keeps only the field along the transmission axis at angle $\theta$ |
| $W_{jk}(\Gamma, \alpha)$ | `wave_plate(retardance, alpha)` | (2, 2) complex | delays the slow axis by $\Gamma$: $\Gamma = \pi/2$ quarter-wave, $\Gamma = \pi$ half-wave |
| $M_{ik} = P_{ij}W_{jk}$ | `einsum('ij,jk->ik', p, w)` | (2, 2) complex | chain elements (repeated $j$ summed); the first element the light meets is on the right |
| $E^{out}_i = M_{ij}E_j$ | `einsum('ij,j->i', m, e)` | (2,) complex | send the light through the optics |
| $I = E^*_i E_i$ | `einsum('i,i->', e.conj(), e).real` | () | repeated $i$ ⇒ summed: intensity |
| $\mathcal E_i(t)$ | `einsum('i,t->ti', e, exp(-1j*wt)).real` | (T, 2) | the real field over one period: it traces the polarization ellipse |

The light starts linearly polarized at the input angle with intensity $I_0 = 1$. The dashed grey curve on the right is Malus's law $I_0\cos^2\theta$ (no wave plate); the blue curve is what passes after the plate. A quarter-wave plate at 45° to the input makes circular light (equal $\lvert E_x\rvert$ and $\lvert E_y\rvert$, 90° out of phase), so every polarizer angle passes $I_0/2$. A half-wave plate at $\alpha$ rotates linear light by $2\alpha$. A single Jones vector is always fully polarized; unpolarized light is handled by averaging the intensities of two orthogonal inputs (used for the three-polarizer check printed by main.py).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/polarization_jones/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    e_in = tensors.linear_input(params["input_angle"])
    w = tensors.plate(params["plate"], params["plate_angle"])
    angles = tensors.analyzer_angles()
    wt = tensors.phases()

    result = solve.solve(e_in, w, angles, wt)

    ex, ey, phase = solve.describe(result["e_plate"])
    print(f"after the {params['plate']} plate: |Ex| = {ex:.3f}, |Ey| = {ey:.3f}, phase(Ey) − phase(Ex) = {phase:.1f}°")
    # classic check: unpolarized light through polarizers at 0°, 45°, 90° passes I₀/8 (I₀/2 after the first one)
    crossed = equation.chain(equation.polarizer(0.0), equation.polarizer(math.pi / 2))
    three = equation.chain(equation.polarizer(0.0), equation.polarizer(math.pi / 4), equation.polarizer(math.pi / 2))
    print(f"unpolarized I0 = 1 through 0°, 90° polarizers:      I = {equation.unpolarized_intensity(crossed):.4f}")
    print(f"unpolarized I0 = 1 through 0°, 45°, 90° polarizers: I = {equation.unpolarized_intensity(three):.4f}")
    return plot.animate(angles, result, params["plate"], out_dir / "polarization_jones.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/polarization_jones/equation.py
r"""Physics → tensor translation.

For light travelling along z the transverse field is \mathcal E_i(t) = \operatorname{Re}(E_i e^{-i\omega t}),
i ∈ {x, y}. The complex 2-vector E_i (the Jones vector) holds the amplitudes and the relative phase
of the two components: in phase ⇒ linear, 90° apart with equal size ⇒ circular, otherwise elliptical.

Linear optical elements act as complex 2×2 matrices. In the element's own axes:

    polarizer:              diag(1, 0)             (keep the field along its axis)
    wave plate, delay Γ:    diag(1, e^{iΓ})        (Γ = π/2 quarter-wave, Γ = π half-wave)

Rotated by an angle θ in the lab:  M(θ) = R(θ)\, M_0\, R(-θ). A sequence of elements is the
matrix product, rightmost first, and the output field and intensity are

    E^{out}_i = P_{ij} W_{jk} E^{in}_k,        I = E^*_i E_i

(repeated j, k and i ⇒ summed). einsum('ij,j->i') applies a matrix, einsum('ij,jk->ik') chains two.
"""

import math

import torch


def rotation(theta: float) -> torch.Tensor:
    """R(θ) = [[cos θ, −sin θ], [sin θ, cos θ]]: shape (2, 2), complex so it multiplies Jones matrices."""
    c, s = math.cos(theta), math.sin(theta)
    return torch.tensor([[c, -s], [s, c]], dtype=torch.complex64)


def rotated(m0: torch.Tensor, theta: float) -> torch.Tensor:
    """M(θ) = R(θ) M₀ R(−θ): the element M₀ turned by θ.   (2, 2) → (2, 2)"""
    return chain(rotation(-theta), m0, rotation(theta))


def polarizer(theta: float) -> torch.Tensor:
    """Linear polarizer with transmission axis at angle θ: (2, 2) complex."""
    return rotated(torch.tensor([[1, 0], [0, 0]], dtype=torch.complex64), theta)


def wave_plate(retardance: float, alpha: float) -> torch.Tensor:
    """Wave plate delaying the slow axis by Γ = retardance, fast axis at angle α: (2, 2) complex."""
    delay = complex(math.cos(retardance), math.sin(retardance))  # e^{iΓ}
    return rotated(torch.tensor([[1, 0], [0, delay]], dtype=torch.complex64), alpha)


def chain(*elements: torch.Tensor) -> torch.Tensor:
    r"""Elements in the order the light meets them → one matrix  M = E_n … E_2 E_1.

    Each step is  M_{ik} ← E_{ij} M_{jk}  (repeated j summed).
    """
    m = torch.eye(2, dtype=torch.complex64)
    for e in elements:
        m = torch.einsum("ij,jk->ik", e, m)
    return m


def apply(m: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""E^{out}_i = M_{ij} E_j.   (2, 2), (2,) → (2,)"""
    return torch.einsum("ij,j->i", m, e)


def intensity(e: torch.Tensor) -> torch.Tensor:
    r"""I = E^*_i E_i (a real number).   (2,) → ()"""
    return torch.einsum("i,i->", e.conj(), e).real


def unpolarized_intensity(m: torch.Tensor) -> torch.Tensor:
    """Unpolarized light of intensity 1 = equal, incoherent mix of x- and y-polarized light (½ each)."""
    ex = torch.tensor([1, 0], dtype=torch.complex64)
    ey = torch.tensor([0, 1], dtype=torch.complex64)
    return 0.5 * (intensity(apply(m, ex)) + intensity(apply(m, ey)))


def field_trace(e: torch.Tensor, wt: torch.Tensor) -> torch.Tensor:
    r"""\mathcal E_i(t) = Re(E_i e^{-iωt}) at the phases ωt: (2,), (T,) → (T, 2). It draws the ellipse."""
    return torch.einsum("i,t->ti", e, torch.exp(-1j * wt)).real


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/polarization_jones/tensors.py
"""Build the concrete tensors: the input Jones vector, the wave plate and the analyzer angles."""

import math

import torch

from .equation import wave_plate

RETARDANCE = {"none": 0.0, "quarter": math.pi / 2, "half": math.pi}  # Γ for each plate choice


def linear_input(angle_deg: float) -> torch.Tensor:
    """Linearly polarized light at the given angle, intensity 1: E = (cos φ, sin φ), shape (2,) complex."""
    phi = math.radians(angle_deg)
    return torch.tensor([math.cos(phi), math.sin(phi)], dtype=torch.complex64)


def plate(kind: str, angle_deg: float) -> torch.Tensor:
    """Jones matrix W of the chosen wave plate (the identity for "none"): (2, 2) complex."""
    return wave_plate(RETARDANCE[kind], math.radians(angle_deg))


def analyzer_angles(n: int = 90) -> torch.Tensor:
    """Angles of the rotating polarizer (analyzer), 0 → 180°, in radians: (A,)."""
    return torch.deg2rad(torch.linspace(0.0, 180.0, n))


def phases(n: int = 100) -> torch.Tensor:
    """ωt over one optical period, used to trace the field ellipse: (T,)."""
    return torch.linspace(0.0, 2 * math.pi, n)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/polarization_jones/solve.py
"""Send the light through the wave plate and the analyzer at every angle; record fields and intensities."""

import torch

from .equation import apply, chain, field_trace, intensity, polarizer


def solve(e_in: torch.Tensor, w: torch.Tensor, angles: torch.Tensor, wt: torch.Tensor) -> dict[str, torch.Tensor]:
    """Return a dict with

    ellipse_in (T, 2), ellipse_plate (T, 2): field traces before and after the plate,
    e_out (A, 2) complex: field after the analyzer at every analyzer angle,
    intensity (A,): transmitted intensity with the plate,  malus (A,): the same without it.
    """
    e_plate = apply(w, e_in)  # (2,) light between plate and analyzer
    e_out, i_out, i_malus = [], [], []
    for theta in angles.tolist():
        p = polarizer(theta)
        m = chain(w, p)  # plate first, then analyzer:  M = P W
        e_out.append(apply(m, e_in))
        i_out.append(intensity(e_out[-1]))
        i_malus.append(intensity(apply(p, e_in)))  # analyzer alone
    return {
        "e_plate": e_plate,
        "ellipse_in": field_trace(e_in, wt),
        "ellipse_plate": field_trace(e_plate, wt),
        "e_out": torch.stack(e_out),
        "intensity": torch.stack(i_out),
        "malus": torch.stack(i_malus),
    }


def describe(e: torch.Tensor) -> tuple[float, float, float]:
    """(|Ex|, |Ey|, phase of Ey relative to Ex in degrees) of a Jones vector."""
    phase = torch.rad2deg(torch.angle(e[1]) - torch.angle(e[0]))
    phase = (phase + 180) % 360 - 180  # wrap into [−180°, 180°)
    return e[0].abs().item(), e[1].abs().item(), phase.item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/polarization_jones/plot.py
"""Animate the analyzer rotating: field ellipse and transmitted field (left), intensity vs angle (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(angles: torch.Tensor, res: dict, plate: str, path: Path) -> Path:
    theta = angles.numpy()  # (A,)
    ell_in, ell_plate = res["ellipse_in"].numpy(), res["ellipse_plate"].numpy()  # (T, 2) each
    i_out, i_malus = res["intensity"].numpy(), res["malus"].numpy()  # (A,) each
    n_phase = len(ell_plate)

    fig, (ax, ax_i) = plt.subplots(1, 2, figsize=(7, 4))
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)
    ax.set_aspect("equal")
    ax.set_xlabel("Ex")
    ax.set_ylabel("Ey")
    ax.plot(ell_in[:, 0], ell_in[:, 1], "--", color="#9ca3af", lw=1.5, label="input")
    ax.plot(ell_plate[:, 0], ell_plate[:, 1], color="#2563eb", lw=1.5, label=f"after {plate} plate")
    (axis_line,) = ax.plot([], [], color="#374151", lw=0.8, ls="-.")
    (passed,) = ax.plot([], [], color="#dc2626", lw=3, label="after analyzer")
    (tip,) = ax.plot([], [], "o", color="#2563eb", ms=6)
    (proj,) = ax.plot([], [], ":", color="#dc2626", lw=1)
    ax.legend(loc="upper right", fontsize=7)
    ax.set_title("field ellipse")

    deg = np.degrees(theta)
    ax_i.plot(deg, i_malus, "--", color="#9ca3af", label="no plate: cos²θ (Malus)")
    ax_i.plot(deg, i_out, color="#2563eb", label=f"with {plate} plate")
    (dot,) = ax_i.plot([], [], "o", color="#dc2626")
    ax_i.set_xlim(0, 180)
    ax_i.set_ylim(0, 1.15)
    ax_i.set_xticks([0, 45, 90, 135, 180])
    ax_i.set_xlabel("analyzer angle θ (°)")
    ax_i.set_ylabel("transmitted I / I₀")
    ax_i.legend(loc="upper right", fontsize=7)
    title = ax_i.set_title(" ")

    def update(a: int):
        u = np.array([np.cos(theta[a]), np.sin(theta[a])])  # analyzer transmission axis
        axis_line.set_data([-1.3 * u[0], 1.3 * u[0]], [-1.3 * u[1], 1.3 * u[1]])
        amp = np.sqrt(i_out[a])  # |E_out|: the passed field oscillates along u with this amplitude
        passed.set_data([-amp * u[0], amp * u[0]], [-amp * u[1], amp * u[1]])
        e_now = ell_plate[(7 * a) % n_phase]  # field at the current instant: it runs around the ellipse
        e_passed = (e_now @ u) * u  # the analyzer keeps the component along its axis
        tip.set_data([e_now[0]], [e_now[1]])
        proj.set_data([e_now[0], e_passed[0]], [e_now[1], e_passed[1]])
        dot.set_data([deg[a]], [i_out[a]])
        title.set_text(f"θ = {deg[a]:.0f}°,  I = {i_out[a]:.2f} I₀")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(theta), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'plate': 'quarter',
    'plate_angle': 45.0,
    'input_angle': 0.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.polarization_jones")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.polarization_jones.main")
problem_dir = Path("problems/optics/polarization_jones")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
